# 07 — AN5 Engagement Sensitivity

This notebook executes the five canonical AN5 SQL investigations through the shared read-only PostgreSQL runner. SQL remains the single source of truth under `sql/04_analytics/`; no query text is duplicated here.

AN5 compares R0 (all source rows), R1 (exact six-column duplicate exclusion for metric calculation only), and R2 (maximum `sum_click` per five-column candidate key for sensitivity only). These are virtual query scenarios, not source-cleaning rules.

The notebook follows the direct-cell execution style of the historical WO-1 notebook: after the read-only preflight passes, each query cell executes directly without an interactive authorization prompt or per-query approval request. Execute cells individually from top to bottom; do not use Run All because each query scans the large activity table.

## 1. Environment setup

This cell locates the repository, imports the shared runner and AN5 helper, and initializes an in-memory evidence dictionary. It does not connect to PostgreSQL.

In [ ]:
# 1. Locate the repository and import the canonical AN5 execution helpers.
from pathlib import Path
import sys
from IPython.display import display

_current = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        candidate
        for candidate in [_current, *_current.parents]
        if (candidate / "src" / "analytics" / "db.py").is_file()
        and (candidate / "sql" / "04_analytics").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open Jupyter from the repository root or notebooks folder.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.run_an5 import (
    AN5_QUERY_IDS,
    EXPECTED_RESULT_SET_COUNTS,
    run_an5_query,
    verify_read_only_preflight,
)

query_results = {}
print("Project root:", PROJECT_ROOT)
print("AN5 query IDs:", AN5_QUERY_IDS)
print("Shared read-only runner imported:", callable(run_an5_query))

## 2. Database identity and read-only preflight

The canonical AN0.01 query verifies the active database and both read-only settings. No AN5 query runs unless this preflight succeeds.

In [ ]:
# 2. Verify the intended database and read-only session before AN5 execution.
preflight = verify_read_only_preflight()
query_results["PREFLIGHT.AN0.01"] = preflight
display(preflight["data"])
print("Preflight elapsed seconds:", preflight["elapsed_seconds"])
print("Preflight: PASS")

## 3. AN5.01 — Scenario and source-grain reconciliation

Returns three result sets: source repetition profile, R0/R1/R2 row-count bases, and activity-to-registration reconciliation. Review source counts, five-column candidate-key behavior, exact six-column duplicates, and unmatched activity.

In [ ]:
# 3. Execute AN5.01 and retain all returned result sets.
result_an501 = run_an5_query("AN5.01")
query_results["AN5.01"] = result_an501
for item in result_an501["result_sets"]:
    print(f"Result set {item['statement_number']}: {item['rows_displayed']} rows; truncated={item['display_truncated']}")
    display(item["data"])
print("Batch seconds:", result_an501["elapsed_seconds"])

## 4. AN5.02 — Student-presentation activity sensitivity

Compares recorded click volume, activity presence, student-active-day pairs, and student-active-site pairs using the same registered student-presentation denominator. Presentation-level and all-presentation summaries are returned.

In [ ]:
# 4. Execute AN5.02 and retain its result set.
result_an502 = run_an5_query("AN5.02")
query_results["AN5.02"] = result_an502
for item in result_an502["result_sets"]:
    print(f"Result set {item['statement_number']}: {item['rows_displayed']} rows; truncated={item['display_truncated']}")
    display(item["data"])
print("Batch seconds:", result_an502["elapsed_seconds"])

## 5. AN5.03 — Temporal activity sensitivity

Separates pre-course, in-course, and post-course activity, while retaining missing course metadata and unmatched registration keys. Relative days are not calendar dates.

In [ ]:
# 5. Execute AN5.03 and retain its result set.
result_an503 = run_an5_query("AN5.03")
query_results["AN5.03"] = result_an503
for item in result_an503["result_sets"]:
    print(f"Result set {item['statement_number']}: {item['rows_displayed']} rows; truncated={item['display_truncated']}")
    display(item["data"])
print("Batch seconds:", result_an503["elapsed_seconds"])

## 6. AN5.04 — Student ranking sensitivity

Ranks registered student-presentations within each module-presentation under each scenario. Reports average-rank Spearman correlation, absolute rank displacement, and top-decile overlap for R0/R1, R0/R2, and R1/R2. Ties at the top-decile boundary are included.

In [ ]:
# 6. Execute AN5.04 and retain its result set.
result_an504 = run_an5_query("AN5.04")
query_results["AN5.04"] = result_an504
for item in result_an504["result_sets"]:
    print(f"Result set {item['statement_number']}: {item['rows_displayed']} rows; truncated={item['display_truncated']}")
    display(item["data"])
print("Batch seconds:", result_an504["elapsed_seconds"])

## 7. AN5.05 — Outcome-group activity sensitivity

Compares activity measures within final-result groups and module-presentations. Final results are retrospective descriptive labels, not predictors. The second result set diagnoses registration-to-profile matching and preserves missing or conflicting profile states.

In [ ]:
# 7. Execute AN5.05 and retain both result sets.
result_an505 = run_an5_query("AN5.05")
query_results["AN5.05"] = result_an505
for item in result_an505["result_sets"]:
    print(f"Result set {item['statement_number']}: {item['rows_displayed']} rows; truncated={item['display_truncated']}")
    display(item["data"])
print("Batch seconds:", result_an505["elapsed_seconds"])

## 8. Evidence inventory and AN5 closeout

A successful SQL call is not AN5 acceptance. Reconcile scenario row counts, student-presentation grain, registration and profile joins, common denominators, temporal buckets, rank stability, top-decile overlap, outcome-group sizes, and result-set truncation. Interpret differences as sensitivity evidence, not proof that one scenario is correct.

Do not approve an engagement KPI, causal claim, or predictive feature from AN5 alone. AN6 remains a separate roadmap stage and may begin only after the project owner reviews the evidence and accepts the activity definition and limitations.

In [ ]:
# 8. Summarize retained execution evidence without inventing findings.
execution_inventory = []
for query_id in AN5_QUERY_IDS:
    result = query_results.get(query_id)
    if result is None:
        execution_inventory.append({"query_id": query_id, "status": "NOT EXECUTED"})
        continue
    execution_inventory.append({
        "query_id": query_id,
        "status": "EXECUTED",
        "result_sets": result["result_set_count"],
        "elapsed_seconds": result["elapsed_seconds"],
        "display_truncated": any(x["display_truncated"] for x in result["result_sets"]),
        "displayed_rows": sum(x["rows_displayed"] for x in result["result_sets"]),
    })

import pandas as pd
display(pd.DataFrame(execution_inventory))